# Karnataka Agriculture Yield Prediction, Financial Risk & Climate Stress Simulation

End-to-end machine learning pipeline analyzing agricultural yield (`Tn/ha`), financial risk (`INR/ha` and `USD/ha`), 2026–2027 climate stress scenarios, and Unit Management Index (UMI) parcel segmentation across 11 locations and 13 crops in Karnataka, India (2004–2019).

In [1]:
# Librerias a utilizar (estandar para analisis y visualizacion)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

In [2]:
# Fuente de datos

# Obteniendo la ruta para la base de datos
import kagglehub

# Descargar dataset
path = kagglehub.dataset_download(
    "imtkaggleteam/agriculture-dataset-karnataka"
)

print("Path to dataset files:", path)

# Obteniendo los datos

from pathlib import Path

# Convertir la ruta en un objeto Path
dataset_path = Path(path)

# Listar archivos disponibles
for file in dataset_path.rglob("*"):
    if file.is_file():
        print(file.name, "|", file.stat().st_size / 1024, "KB")

100%|██████████| 41.2k/41.2k [00:00<00:00, 32.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/imtkaggleteam/agriculture-dataset-karnataka/versions/1
data_season.csv | 218.7578125 KB


In [3]:
# Comenzando exploracion/entendimiento de datos

#Vamos a ver cuantos archivos se tiene en el archivo de datos
csv_files = list(dataset_path.rglob("*.csv"))

# Mostrar lo encontrado
print(csv_files)

# Entonces se procederá a generar finalmente el dataframe

df = pd.read_csv(csv_files[0])

# Vamos a visualizar las primeras 10 filas
print("\nTabla de las primeras 10 filas")
print(df.head(10))

# Vamos a conseguir un resumen de los datos
print("\nResumen de la base de datos")
print(df.info())

# Vamos a conseguir el estadístico descriptivo
print("\nEstadística descriptiva")
print(df.describe())

# vamos a dimensionar nuestro df
print("\nDimensiones del df")
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

# Vamos a encontrar numero de valores nulos y duplicados
print("\nValores nulos")
print(df.isnull().sum())

print("\nValores duplicados")
print(df.duplicated().sum())

[PosixPath('/root/.cache/kagglehub/datasets/imtkaggleteam/agriculture-dataset-karnataka/versions/1/data_season.csv')]

Tabla de las primeras 10 filas
   Year   Location   Area  Rainfall  Temperature Soil type Irrigation  \
0  2004  Mangalore   1279    2903.1         27.0  Alluvial       Drip   
1  2004  Mangalore  13283    2903.1         27.0  Alluvial       Drip   
2  2004  Mangalore  52119    2903.1         27.0  Alluvial       Drip   
3  2004  Mangalore    725    2996.6         27.0  Alluvial       Drip   
4  2004  Mangalore  12885    2996.6         27.0  Alluvial       Drip   
5  2004  Mangalore  51179    2996.6         27.0  Alluvial       Drip   
6  2004  Mangalore    468    3522.8         27.2  Alluvial       Drip   
7  2004  Mangalore  11841    3522.8         27.2  Alluvial       Drip   
8  2004  Mangalore  50975    3522.8         27.2  Alluvial       Drip   
9  2004  Mangalore    206    3729.8         27.0  Alluvial       Drip   

     yeilds  Humidity    Crops   price  Season

In [4]:
# Observaciones sobre la columna crops y otras para empezar a limpiar los datos
## EDA y procesamiento/adaptación de datos

# Filtra por un cultivo específico para ver si los números tienen sentido como kg/ha
# print(df[df['Crops'] == 'Coconut'][['Area', 'yeilds']].head(10))

# Para comenzar con la limpieza se creará una copia del df
df_copy = df.copy()

# Se cambiara el nombre de yeilds por produccion total para tener mejor claridad de lo que representa la columna
df_copy = df_copy.rename(columns={'yeilds': 'Produccion_Total(Tn)'})
df_copy = df_copy.rename(columns={'Area': 'Area(ha)'})
df_copy = df_copy.rename(columns={'Temperature': 'Temperature(C°)'})
df_copy = df_copy.rename(columns={'Rainfall': 'Rainfall(mm)'})
df_copy = df_copy.rename(columns={'Humidity': 'Humidity(%)'})
# Vamos a crear una nueva columna donde se refleje el verdadero valor de yield por area
df_copy['yield_per_area(Tn/ha)'] = df_copy['Produccion_Total(Tn)'] / df_copy['Area(ha)']

print(df_copy.head(10))



   Year   Location  Area(ha)  Rainfall(mm)  Temperature(C°) Soil type  \
0  2004  Mangalore      1279        2903.1             27.0  Alluvial   
1  2004  Mangalore     13283        2903.1             27.0  Alluvial   
2  2004  Mangalore     52119        2903.1             27.0  Alluvial   
3  2004  Mangalore       725        2996.6             27.0  Alluvial   
4  2004  Mangalore     12885        2996.6             27.0  Alluvial   
5  2004  Mangalore     51179        2996.6             27.0  Alluvial   
6  2004  Mangalore       468        3522.8             27.2  Alluvial   
7  2004  Mangalore     11841        3522.8             27.2  Alluvial   
8  2004  Mangalore     50975        3522.8             27.2  Alluvial   
9  2004  Mangalore       206        3729.8             27.0  Alluvial   

  Irrigation  Produccion_Total(Tn)  Humidity(%)    Crops   price  Season  \
0       Drip                2570.0         57.0  Coconut  200000  Kharif   
1       Drip               27170.0         5

# Haremos un pequeño ejercicio de clasificación durante la siguiente sección.

In [5]:
# Crearemos un dataframe filtrado donde se contengan los datos No nulos y otro donde se contengan los datos nulos del tipo de suelo
df_copy_no_null = df_copy.dropna()
df_copy_null = df_copy[df_copy['Soil type'].isnull()]

# vamos a ver el resumen de ambas bases
print("\nValores no nulos")
print(df_copy_no_null.info())

print("\nValores nulos")
print(df_copy_null.info())

# Vamos a asignar las variables de entrenamiento
X_train = df_copy_no_null.drop(columns=["Soil type"])
y_train = df_copy_no_null["Soil type"]

# Vamos a asignar las variables de prueba
X_test = df_copy_null.drop(columns=["Soil type"])



Valores no nulos
<class 'pandas.core.frame.DataFrame'>
Index: 3100 entries, 0 to 3099
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Year                   3100 non-null   int64  
 1   Location               3100 non-null   object 
 2   Area(ha)               3100 non-null   int64  
 3   Rainfall(mm)           3100 non-null   float64
 4   Temperature(C°)        3100 non-null   float64
 5   Soil type              3100 non-null   object 
 6   Irrigation             3100 non-null   object 
 7   Produccion_Total(Tn)   3100 non-null   float64
 8   Humidity(%)            3100 non-null   float64
 9   Crops                  3100 non-null   object 
 10  price                  3100 non-null   int64  
 11  Season                 3100 non-null   object 
 12  yield_per_area(Tn/ha)  3100 non-null   float64
dtypes: float64(5), int64(3), object(5)
memory usage: 339.1+ KB
None

Valores nulos
<class 'pandas.c

In [6]:
# Vamos a colocar la libreria a utilizar para este entrenamiento
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [7]:
# Identificar columnas categóricas y codificar el dataset completo para evitar desalineación de características
# Primero eliminamos la columna objetivo 'Soil type' para la codificación de variables independientes
df_features = df_copy.drop(columns=["Soil type"])
X_encoded_all = pd.get_dummies(df_features, drop_first=True)

# Ahora separamos en base a los índices que tenían nulos o no nulos en 'Soil type'
X_encoded_no_null = X_encoded_all.loc[df_copy_no_null.index]
X_test_encoded = X_encoded_all.loc[df_copy_null.index]

y_encoded = df_copy_no_null["Soil type"]

# Dividir los datos de entrenamiento y validación con los datos codificados
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded_no_null, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Entrenando modelo Random Forest
model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# Finalmente evaluamos el modelo con el conjunto de validación
y_pred_val = model.predict(X_val)
print("Accuracy en Validación:", accuracy_score(y_val, y_pred_val))
print("Classification Report de Validación:\n", classification_report(y_val, y_pred_val))

# Ahora pasaremos a la prueba de fuego prediciendo los valores nulos (X_test_encoded)
y_pred_test = model.predict(X_test_encoded)
print("\nPredicciones exitosas para las filas con valores nulos en Soil type!")
print("Primeras 10 predicciones para los valores faltantes:", y_pred_test[:10])

# Insertando valores
predictions = df_copy_null.copy()
predictions["Soil type"] = y_pred_test
df_copy_no_null_T = pd.concat([df_copy_no_null, predictions])

print('\nMostrando el ultimo df:')
print(df_copy_no_null_T.info())

Accuracy en Validación: 0.8951612903225806
Classification Report de Validación:
                       precision    recall  f1-score   support

            Alluvial       0.85      0.92      0.89        92
     Arid and Desert       0.93      1.00      0.96        13
               Black       0.92      0.90      0.91        52
        Black cotton       0.83      1.00      0.91         5
                Clay       0.89      0.81      0.85        21
           Clay loam       0.85      0.85      0.85        39
        Drained loam       0.93      0.76      0.84        17
          Dry sandy        0.95      0.95      0.95        20
       Gravelly sand       1.00      1.00      1.00         8
          Heavy clay       1.00      1.00      1.00         8
        Heavy cotton       1.00      0.83      0.91         6
            Laterite       0.91      0.88      0.89        56
         Light sandy       1.00      1.00      1.00         6
                Loam       0.85      0.95      0.9

# Ahora con el ejercicio de clasificacion terminado y con nuestro nuevo df "df_copy_null_T" podremos hacer el caso de regresión.

In [8]:
# Comenzaremos renombrando el df por fines de comodidad
df_regresion = df_copy_no_null_T

# Ordenaremos los datos
df_regresion = df_regresion.sort_values(by=['Year', 'Location', 'Crops']).reset_index(drop=True)

### 1. EDA no.2
Iniciamos con la exploración profunda de datos

In [9]:
# Iniciand exploración

## Cobertura temporal por cultivo y ubicación.

cobertura = (df_regresion.groupby(['Location', 'Crops'])['Year'].agg(
    first_year = 'min',
    last_year = 'max',
    years_observed = 'nunique').reset_index()
    )

cobertura['years_missing'] = (16 - cobertura['years_observed'])
display(cobertura.sort_values(['years_missing', 'Location'], ascending=[False, True]))

# Número de observaciones por año

display(
    df_regresion.groupby('Year')
    .agg(
        observations=('Crops', 'size'),
        locations=('Location', 'nunique'),
        crops=('Crops', 'nunique')
    )
    .sort_index()
)


         Location    Crops  first_year  last_year  years_observed  \
4   Chikmangaluru   Cashew        2004       2004               1   
5   Chikmangaluru    Cocoa        2010       2010               1   
7   Chikmangaluru   Cotton        2004       2004               1   
29       Gulbarga      Tea        2004       2004               1   
37         Hassan   Cotton        2004       2004               1   
..            ...      ...         ...        ...             ...   
38         Hassan   Ginger        2004       2019              16   
41         Hassan   Pepper        2004       2019              16   
48     Kasaragodu  Coconut        2004       2019              16   
73      Mangalore  Coconut        2004       2019              16   
81         Mysuru  Coconut        2004       2019              16   

    years_missing  
4              15  
5              15  
7              15  
29             15  
37             15  
..            ...  
38              0  
41         

      observations  locations  crops
Year                                
2004           914         10     13
2005           132         10     12
2006           126         10     12
2007           126         10     12
2008           126         10     13
2009           126         11     13
2010           126         11     13
2011           132         11     13
2012           144         11     13
2013           186         11     12
2014           186         11     13
2015           192         11     13
2016           216         11     13
2017           222         11     13
2018           186         11     12
2019            18          6      8

## El siguiente bloque de codigo es para el desarrollo de funciones que pueden o no ser de utilidad y se iran descartando de acuerdo a la necesidad el ejercicio.

In [10]:
# Funciones

def analizar_interacciones(df, cultivo, tipo_suelo):
    """
    Filtra el dataset por un cultivo y tipo de suelo específicos para
    mostrar la matriz de correlación completa entre las variables ambientales y el rendimiento.
    """
    # Filtrar datos
    df_filtrado = df[(df['Crops'] == cultivo) & (df['Soil type'] == tipo_suelo)]

    if len(df_filtrado) < 5:
        print(f"⚠️ Hay muy pocos datos ({len(df_filtrado)} registros) para la combinación: {cultivo} en suelo {tipo_suelo}.")
        return

    # Seleccionar variables de interés
    vars_interes = ['yield_per_area(Tn/ha)', 'Rainfall(mm)', 'Temperature(C°)', 'Humidity(%)']
    corr_matrix = df_filtrado[vars_interes].corr()

    # Graficar el mapa de calor
    plt.figure(figsize=(7, 5))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".3f")
    plt.title(f"Interacciones Climáticas para {cultivo} en Suelo '{tipo_suelo}'\n(Registros: {len(df_filtrado)})")
    plt.tight_layout()
    plt.show()


def analizar_correlaciones(df, cultivo, suelo=None):

    variables = [
        'yield_per_area(Tn/ha)',
        'Rainfall(mm)',
        'Temperature(C°)',
        'Humidity(%)'
    ]

    # Filtrar cultivo
    datos = df[df['Crops'] == cultivo].copy()

    # Filtrar suelo si se especifica
    if suelo is not None:
        datos = datos[datos['Soil type'] == suelo]

    # Limpiar datos
    datos = datos[variables].dropna()

    n = len(datos)

    print(f"Cultivo: {cultivo}")
    print(f"Suelo: {suelo if suelo else 'Todos'}")
    print(f"Observaciones: {n}")

    if n < 3:
        print("Datos insuficientes para calcular correlaciones.")
        return None

    resultados = []

    objetivo = 'yield_per_area(Tn/ha)'

    for variable in variables[1:]:

        # Comprobar variabilidad
        if datos[variable].nunique() < 2 or datos[objetivo].nunique() < 2:

            resultados.append({
                'Variable': variable,
                'Pearson': None,
                'P-value Pearson': None,
                'Spearman': None,
                'P-value Spearman': None
            })

            continue

        # Pearson
        pearson, p_pearson = pearsonr(
            datos[variable],
            datos[objetivo]
        )

        # Spearman
        spearman, p_spearman = spearmanr(
            datos[variable],
            datos[objetivo]
        )

        resultados.append({
            'Variable': variable,
            'Pearson': pearson,
            'P-value Pearson': p_pearson,
            'Spearman': spearman,
            'P-value Spearman': p_spearman
        })

    # IMPORTANTE: return fuera del ciclo for
    return pd.DataFrame(resultados)

In [11]:
# Descartando la posibilidad de datos nulos
for year in [2004, 2018, 2019]:
    print(f"\n========== AÑO {year} ==========")
    resumen = (
        df_regresion[df_regresion['Year'] == year]
        .groupby(['Location', 'Crops'])
        .size()
        .reset_index(name='observations')
        .sort_values('observations', ascending=False)
    )
    display(resumen.head(10))


print("Duplicados exactos:", df_regresion.duplicated().sum())

repeticiones = (
    df_regresion
    .groupby(['Year', 'Location', 'Crops'])
    .size()
    .reset_index(name='observations')
)

print("\nCombinaciones con más de una observación:")
display(
    repeticiones[repeticiones['observations'] > 1]
    .sort_values('observations', ascending=False)
    .head(15)
)


========== AÑO 2004 ==========


         Location    Crops  observations
16       Gulbarga  Coconut           108
3   Chikmangaluru  Coconut            85
28         Hassan  Coconut            70
74        Raichur  Coconut            62
5   Chikmangaluru   Ginger            35
52       Madikeri  Coconut            32
46         Kodagu  Coconut            30
39     Kasaragodu  Coconut            27
62      Mangalore  Coconut            25
18       Gulbarga   Ginger            23


========== AÑO 2018 ==========


         Location    Crops  observations
25         Hassan  Coconut            20
15       Gulbarga  Coconut            13
53      Mangalore  Coconut            11
60         Mysuru  Coconut            10
34     Kasaragodu  Coconut             8
39         Kodagu  Coconut             6
3   Chikmangaluru  Coconut             6
27         Hassan   Ginger             5
69        Raichur  Coconut             5
45       Madikeri  Coconut             5


========== AÑO 2019 ==========


     Location      Crops  observations
8   Mangalore    Coconut             4
1      Hassan    Coconut             3
0      Hassan  Blackgram             1
3      Hassan     Pepper             1
2      Hassan     Ginger             1
4  Kasaragodu    Coconut             1
5      Kodagu   Arecanut             1
6      Kodagu   Cardamum             1
7      Kodagu     Pepper             1
9   Mangalore     Ginger             1

Duplicados exactos: 0

Combinaciones con más de una observación:


     Year       Location    Crops  observations
16   2004       Gulbarga  Coconut           108
3    2004  Chikmangaluru  Coconut            85
28   2004         Hassan  Coconut            70
74   2004        Raichur  Coconut            62
5    2004  Chikmangaluru   Ginger            35
52   2004       Madikeri  Coconut            32
46   2004         Kodagu  Coconut            30
39   2004     Kasaragodu  Coconut            27
62   2004      Mangalore  Coconut            25
761  2015         Hassan  Coconut            23
18   2004       Gulbarga   Ginger            23
602  2013         Hassan  Coconut            23
681  2014         Hassan  Coconut            23
531  2012         Hassan  Coconut            22
915  2017         Hassan  Coconut            21

In [12]:
# Vamos a descrubir la diferencia entre filas
# 1. Ver todas las columnas del dataset
print("Columnas:", df_regresion.columns.tolist())

# 2. Inspeccionar las primeras 10 filas de Gulbarga + Coconut en 2004
muestra_2004 = df_regresion[
    (df_regresion['Year'] == 2004) &
    (df_regresion['Location'] == 'Gulbarga') &
    (df_regresion['Crops'] == 'Coconut')
]
display(muestra_2004.head(10))

# 3. Ver las 18 filas completas de 2019
display(df_regresion[df_regresion['Year'] == 2019])

# Buscando mas datos erroneos o extraños
# Revisar rangos mínimos y máximos de todas las columnas numéricas
display(df_regresion.describe())

# Ver todas las filas con temperaturas mayores a 50°C (imposibles en agricultura)
display(df_regresion[df_regresion['Temperature(C°)'] > 50])

Columnas: ['Year', 'Location', 'Area(ha)', 'Rainfall(mm)', 'Temperature(C°)', 'Soil type', 'Irrigation', 'Produccion_Total(Tn)', 'Humidity(%)', 'Crops', 'price', 'Season', 'yield_per_area(Tn/ha)']


     Year  Location  Area(ha)  Rainfall(mm)  Temperature(C°) Soil type  \
228  2004  Gulbarga         6        3605.4             27.1       Red   
229  2004  Gulbarga        15        3605.4             27.1       Red   
230  2004  Gulbarga         4        3605.4             27.1       Red   
231  2004  Gulbarga        39        3512.4             26.8       Red   
232  2004  Gulbarga         6        3512.4             26.8       Red   
233  2004  Gulbarga        12        3512.4             26.8       Red   
234  2004  Gulbarga         3        3296.0             37.0       Red   
235  2004  Gulbarga        17        3296.0             37.0     Black   
236  2004  Gulbarga         3        3296.0             37.0     Black   
237  2004  Gulbarga         9        3550.0             38.0     Black   

    Irrigation  Produccion_Total(Tn)  Humidity(%)    Crops   price Season  \
228      Basin                  19.0         57.0  Coconut   64409   Rabi   
229      Basin                 

      Year    Location  Area(ha)  Rainfall(mm)  Temperature(C°)  \
3140  2019      Hassan        96        3105.0             27.0   
3141  2019      Hassan      5974        2996.6             40.0   
3142  2019      Hassan      2188        3729.8             37.0   
3143  2019      Hassan        82        3654.0             27.0   
3144  2019      Hassan       206        3654.0             40.0   
3145  2019      Hassan     11841        2996.6             27.0   
3146  2019  Kasaragodu      1298        3605.4            184.0   
3147  2019      Kodagu        53         233.0             27.1   
3148  2019      Kodagu        95        3360.0             27.0   
3149  2019      Kodagu      4954         236.0             27.1   
3150  2019   Mangalore     11841        3522.8             27.0   
3151  2019   Mangalore     51390         234.0             27.0   
3152  2019   Mangalore       819         234.0             27.0   
3153  2019   Mangalore      1468        3522.8             41.

              Year      Area(ha)  Rainfall(mm)  Temperature(C°)  \
count  3158.000000   3158.000000   3158.000000      3158.000000   
mean   2009.929702   9329.913553   2813.126662        65.893699   
std       5.100895  15180.465396   1163.829953        60.814037   
min    2004.000000      1.000000    233.000000        26.800000   
25%    2004.000000    153.000000   2903.100000        27.000000   
50%    2010.000000   2127.000000   3296.000000        38.000000   
75%    2015.000000  10910.000000   3550.000000        43.000000   
max    2019.000000  52119.000000   3729.800000       222.000000   

       Produccion_Total(Tn)  Humidity(%)          price  yield_per_area(Tn/ha)  
count           3158.000000  3158.000000    3158.000000            3158.000000  
mean           23258.976916    55.582331   86271.279291             608.221776  
std            58228.985033     2.469773   94123.160706            5685.046735  
min                2.000000    50.000000     637.000000               0.

      Year       Location  Area(ha)  Rainfall(mm)  Temperature(C°)  \
52    2004  Chikmangaluru     16224        2996.6            221.0   
53    2004  Chikmangaluru     17095        2996.6            222.0   
204   2004      Davangere      6470         234.0            198.0   
205   2004      Davangere     51040        3360.0            222.0   
206   2004       Gulbarga      1374        2903.1            198.0   
...    ...            ...       ...           ...              ...   
3135  2018        Raichur         4        3296.0            142.0   
3136  2018        Raichur       211        3654.0            148.0   
3138  2018        Raichur      2090        3729.8            172.0   
3139  2018        Raichur      1998        2996.6            166.0   
3146  2019     Kasaragodu      1298        3605.4            184.0   

            Soil type Irrigation  Produccion_Total(Tn)  Humidity(%)  \
52         Dry sandy       Basin                  19.0         59.5   
53         Dry sa

In [13]:
#Limpiando los datos
df_limpio = df_regresion.copy()

# 1. Corregir temperaturas imposibles (> 50°C) usando la mediana por ubicación
df_limpio.loc[df_limpio['Temperature(C°)'] > 50, 'Temperature(C°)'] = np.nan
df_limpio['Temperature(C°)'] = df_limpio.groupby('Location')['Temperature(C°)'].transform(
    lambda x: x.fillna(x.median())
)

# 2. Ver los percentiles de rendimiento para cortar los errores de Area = 1
print("Percentiles de rendimiento (Tn/ha):")
print(df_limpio['yield_per_area(Tn/ha)'].quantile([0.50, 0.75, 0.90, 0.95, 0.99]))

# 3. Filtrar outliers extremos (nos quedamos con el 90% de datos físicamente reales)
limite_superior = df_limpio['yield_per_area(Tn/ha)'].quantile(0.90)
df_limpio = df_limpio[df_limpio['yield_per_area(Tn/ha)'] <= limite_superior]

print(f"\nFilas originales: {len(df_regresion)} | Filas limpias: {len(df_limpio)}")
display(df_limpio[['Temperature(C°)', 'yield_per_area(Tn/ha)']].describe())

Percentiles de rendimiento (Tn/ha):
0.50        1.665300
0.75       19.928526
0.90      244.213333
0.95     1126.415556
0.99    12149.078667
Name: yield_per_area(Tn/ha), dtype: float64

Filas originales: 3158 | Filas limpias: 2842


       Temperature(C°)  yield_per_area(Tn/ha)
count      2827.000000            2842.000000
mean         31.474602              14.824968
std           6.070290              36.430171
min          26.800000               0.000039
25%          27.000000               0.099394
50%          27.200000               0.829617
75%          38.000000               8.669053
max          43.000000             243.190476

# Entrenamiento no.2


In [14]:
#Librerias
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [16]:
# 1. Rellenar los 15 nulos restantes de temperatura con la mediana global
df_limpio['Temperature(C°)'] = df_limpio['Temperature(C°)'].fillna(df_limpio['Temperature(C°)'].median())

# 2. Crear variables de dinero (Feature Engineering)
df_limpio['Income_Total'] = df_limpio['Produccion_Total(Tn)'] * df_limpio['price']
df_limpio['Income_per_ha'] = df_limpio['yield_per_area(Tn/ha)'] * df_limpio['price']

# 3. Diagnóstico: Ranking de los cultivos con MENOR rendimiento e ingreso por hectárea
peores_cultivos = (
    df_limpio.groupby('Crops')[['yield_per_area(Tn/ha)', 'Income_per_ha', 'price']]
    .median()
    .sort_values('Income_per_ha', ascending=True)
)
print("=== CULTIVOS CON MENOR INGRESO POR HECTÁREA (Mediana) ===")
display(peores_cultivos)

# 4. Diagnóstico: Top 10 combinaciones (Ubicación + Cultivo) con MAYOR RIESGO (menor ingreso/ha)
zonas_riesgo = (
    df_limpio.groupby(['Location', 'Crops'])[['yield_per_area(Tn/ha)', 'Income_per_ha']]
    .median()
    .sort_values('Income_per_ha', ascending=True)
    .head(10)
)
print("\n=== TOP 10 ZONAS DE ALTO RIESGO (Menor Ingreso/ha) ===")
display(zonas_riesgo)


=== CULTIVOS CON MENOR INGRESO POR HECTÁREA (Mediana) ===


           yield_per_area(Tn/ha)  Income_per_ha    price
Crops                                                   
Cocoa                   0.222222    2345.208120  45186.0
Blackgram               0.200429    5395.466151  43562.0
Paddy                   0.360341   17646.137011  45186.0
Cashew                  0.689320   17845.224544  45186.0
Coffee                  0.612334   18489.353068  43562.0
Cardamum                0.480088   19512.551411  44235.0
Coconut                 1.436040   25593.972251  45186.0
Pepper                  0.547104   28616.503998  45186.0
Tea                     1.047160   31454.661219  45186.0
Ginger                  2.195748   35385.062949  44235.0
Groundnut               2.049619   48119.786723  51239.0
Cotton                  6.993100   58954.527762  43898.5
Arecanut                1.664884   68318.666667  51239.0


=== TOP 10 ZONAS DE ALTO RIESGO (Menor Ingreso/ha) ===


                      yield_per_area(Tn/ha)  Income_per_ha
Location   Crops                                          
Kasaragodu Cocoa                   0.001611     123.990691
Mangalore  Cocoa                   0.028817     345.504413
Davangere  Groundnut               0.145414     370.829588
Raichur    Blackgram               0.173927     791.405737
Kasaragodu Cardamum                0.041788    1017.178200
Hassan     Cocoa                   0.016376    2038.820513
Mangalore  Pepper                  0.187221    2264.656394
Kasaragodu Cashew                  0.050013    2394.384245
Davangere  Blackgram               0.124575    2701.098074
Mangalore  Blackgram               0.173279    3416.861073

In [18]:
# Haciendo algunos gráficos
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Ranking de cultivos (Menor a Mayor)
df_cultivos_plot = peores_cultivos.reset_index()
sns.barplot(
    data=df_cultivos_plot,
    x='Income_per_ha',
    y='Crops',
    ax=axes[0],
    palette='Reds_r'
)
axes[0].set_title('Ingreso Mediano por Hectárea según Cultivo')
axes[0].set_xlabel('Ingreso por Hectárea')
axes[0].set_ylabel('Cultivo')

# Gráfica 2: Top 10 Zonas de Alto Riesgo
df_zonas_plot = zonas_riesgo.reset_index()
df_zonas_plot['Zona_Cultivo'] = df_zonas_plot['Location'] + ' - ' + df_zonas_plot['Crops']

sns.barplot(
    data=df_zonas_plot,
    x='Income_per_ha',
    y='Zona_Cultivo',
    ax=axes[1],
    palette='Oranges_r'
)
axes[1].set_title('Top 10 Zonas de Alto Riesgo (Menor Ingreso/ha)')
axes[1].set_xlabel('Ingreso por Hectárea')
axes[1].set_ylabel('Ubicación - Cultivo')

plt.tight_layout()
plt.show()

/tmp/ipykernel_11016/3059384842.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_11016/3059384842.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


<Figure size 1600x600 with 2 Axes>

In [20]:
# Comenzando entrenamiento numero 2

# 1. Definir X (excluimos price y columnas de producción/dinero para evitar fugas)
columnas_excluir = ['Year', 'price', 'yield_per_area(Tn/ha)', 'Produccion_Total(Tn)', 'Income_Total', 'Income_per_ha']
X_yield = df_limpio.drop(columns=columnas_excluir)
y_yield = df_limpio['yield_per_area(Tn/ha)']

# 2. Convertir texto a columnas numéricas
X_yield = pd.get_dummies(X_yield, drop_first=True)

# 3. División temporal (2004-2017 para entrenar vs 2018 para probar)
filtro_train = df_limpio['Year'] <= 2017
filtro_test = df_limpio['Year'] == 2018

X_train, y_train = X_yield[filtro_train], y_yield[filtro_train]
X_test, y_test = X_yield[filtro_test], y_yield[filtro_test]

# 4. Entrenar modelo de rendimiento físico
modelo_yield = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_yield.fit(X_train, y_train)

# 5. Predecir rendimiento y calcular el ingreso multiplicando por el precio real de 2018
pred_yield_2018 = modelo_yield.predict(X_test)
precios_2018 = df_limpio.loc[filtro_test, 'price']
pred_income_2018 = pred_yield_2018 * precios_2018
real_income_2018 = df_limpio.loc[filtro_test, 'Income_per_ha']

# 6. Evaluar métricas físicas y financieras
print(f"--- RENDIMIENTO FÍSICO ---")
print(f"MAE Rendimiento: {mean_absolute_error(y_test, pred_yield_2018):.2f} Tn/ha")
print(f"R2 Rendimiento: {r2_score(y_test, pred_yield_2018):.2f}")

print(f"\n--- INGRESO CALCULADO (Predicción x Precio) ---")
print(f"MAE Ingreso: ${mean_absolute_error(real_income_2018, pred_income_2018):,.2f} / ha")

--- RENDIMIENTO FÍSICO ---
MAE Rendimiento: 15.28 Tn/ha
R2 Rendimiento: 0.23

--- INGRESO CALCULADO (Predicción x Precio) ---
MAE Ingreso: $1,804,437.33 / ha


In [22]:
# Comenzando entrenamiento numero 3 para pulimiento de modelo (Transformación Logarítmica)

# 1. Comprimir y_train con logaritmo
y_train_log = np.log1p(y_train)

# 2. Entrenar el mismo Random Forest pero sobre la escala comprimida
modelo_yield_log = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_yield_log.fit(X_train, y_train_log)

# 3. Predecir y revertir el logaritmo (expm1) para volver a Tn/ha reales
pred_log_2018 = modelo_yield_log.predict(X_test)
pred_yield_2018_v3 = np.expm1(pred_log_2018)

# 4. Evaluar mejora en rendimiento e ingreso
pred_income_2018_v3 = pred_yield_2018_v3 * precios_2018

print(f"--- RENDIMIENTO FÍSICO (V3) ---")
print(f"MAE Rendimiento: {mean_absolute_error(y_test, pred_yield_2018_v3):.2f} Tn/ha")
print(f"R2 (escala real): {r2_score(y_test, pred_yield_2018_v3):.2f}")
print(f"R2 (escala logarítmica): {r2_score(np.log1p(y_test), pred_log_2018):.2f}")

print(f"\n--- INGRESO CALCULADO (V3) ---")
print(f"MAE Ingreso: ${mean_absolute_error(real_income_2018, pred_income_2018_v3):,.2f} / ha")

--- RENDIMIENTO FÍSICO (V3) ---
MAE Rendimiento: 11.65 Tn/ha
R2 (escala real): 0.25
R2 (escala logarítmica): 0.59

--- INGRESO CALCULADO (V3) ---
MAE Ingreso: $1,470,822.29 / ha


In [23]:
# Pulimiento No. 3 para el entrenamiento del modelo

from sklearn.ensemble import ExtraTreesRegressor

# 1. Limpiar espacios invisibles en Soil type
df_limpio['Soil type'] = df_limpio['Soil type'].str.strip().str.lower()

# 2. Volver a preparar X con la columna limpia
columnas_excluir = ['Year', 'price', 'yield_per_area(Tn/ha)', 'Produccion_Total(Tn)', 'Income_Total', 'Income_per_ha']
X_yield = pd.get_dummies(df_limpio.drop(columns=columnas_excluir), drop_first=True)
y_yield = df_limpio['yield_per_area(Tn/ha)']

filtro_train = df_limpio['Year'] <= 2017
filtro_test = df_limpio['Year'] == 2018

X_train, y_train = X_yield[filtro_train], y_yield[filtro_train]
X_test, y_test = X_yield[filtro_test], y_yield[filtro_test]

# 3. Entrenar ExtraTrees sobre escala logarítmica
modelo_et = ExtraTreesRegressor(n_estimators=300, max_depth=12, min_samples_leaf=2, random_state=42)
modelo_et.fit(X_train, np.log1p(y_train))

pred_log = modelo_et.predict(X_test)
pred_real = np.expm1(pred_log)

print(f"MAE Rendimiento: {mean_absolute_error(y_test, pred_real):.2f} Tn/ha")
print(f"R2 (escala real): {r2_score(y_test, pred_real):.2f}")
print(f"R2 (escala logarítmica): {r2_score(np.log1p(y_test), pred_log):.2f}")

MAE Rendimiento: 12.07 Tn/ha
R2 (escala real): 0.23
R2 (escala logarítmica): 0.59


In [26]:
# Obteniendo las variables de mayor importancia para las predicciones.
# Extraer la importancia de cada variable del modelo ganador (Entrenamiento #3)
importancias = pd.Series(
    modelo_yield_log.feature_importances_,
    index=modelo_yield_log.feature_names_in_
).sort_values(ascending=False)

print("=== TOP 10 VARIABLES QUE DEFINEN EL RENDIMIENTO ===")
print((importancias.head(10) * 100).round(2).astype(str) + '%')

=== TOP 10 VARIABLES QUE DEFINEN EL RENDIMIENTO ===
Area(ha)              50.07%
Rainfall(mm)           6.79%
Humidity(%)            5.81%
Temperature(C°)        4.74%
Crops_Coconut          4.55%
Irrigation_Spray       3.11%
Irrigation_Drip        2.58%
Soil type_Red          1.73%
Soil type_Black        1.33%
Location_Mangalore     1.12%
dtype: object


In [27]:
# Entrenamiento No. 4: usando los datos hasta 2019

# Paso previo a proyecciones 2026-2027: Entrenar modelo final con todo el historial (2004-2019)

# 1. Asegurar limpieza de espacios en Soil type
df_limpio['Soil type'] = df_limpio['Soil type'].str.strip().str.lower()

# 2. Preparar X e y completos (sin filtro de año)
columnas_excluir = ['Year', 'price', 'yield_per_area(Tn/ha)', 'Produccion_Total(Tn)', 'Income_Total', 'Income_per_ha']
X_final = pd.get_dummies(df_limpio.drop(columns=columnas_excluir), drop_first=True)
y_final_log = np.log1p(df_limpio['yield_per_area(Tn/ha)'])

# 3. Entrenar modelo ganador con todos los datos limpios
modelo_final_log = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_final_log.fit(X_final, y_final_log)

# 4. Construir la línea base histórica (medianas por Ubicación y Cultivo) para simular 2026-2027
linea_base = df_limpio.groupby(['Location', 'Crops'], as_index=False).agg({
    'Area(ha)': 'median',
    'Rainfall(mm)': 'median',
    'Temperature(C°)': 'median',
    'Humidity(%)': 'median',
    'price': 'median',
    'Soil type': lambda x: x.mode()[0],
    'Irrigation': lambda x: x.mode()[0],
    'Season': lambda x: x.mode()[0]
})

print(f"Modelo final entrenado con {len(X_final)} registros.")
print(f"Combinaciones (Ubicación + Cultivo) listas para simular: {len(linea_base)}")
display(linea_base.head())

Modelo final entrenado con 2842 registros.
Combinaciones (Ubicación + Cultivo) listas para simular: 98


        Location      Crops  Area(ha)  Rainfall(mm)  Temperature(C°)  \
0      Bangalore  Groundnut     123.0       3512.40             27.2   
1      Bangalore      Paddy     487.5       3482.70             27.2   
2  Chikmangaluru   Arecanut    6316.0       3522.80             27.0   
3  Chikmangaluru  Blackgram    6446.0       3296.00             38.0   
4  Chikmangaluru     Cashew      19.0       3004.05             27.0   

   Humidity(%)     price     Soil type Irrigation  Season  
0         55.0   51239.0   light sandy      Spray    Zaid  
1         56.5  122593.0         black      Spray    Zaid  
2         56.5   54249.0  drained loam      Basin    Rabi  
3         55.5   43436.5      alluvial       Drip  Kharif  
4         55.5   51433.5     clay loam       Drip    Rabi  

In [28]:
# Paso 7: Simulación de rendimiento e ingresos (Income_per_ha) para 2026 y 2027

def proyectar_escenario(df_base, factor_lluvia, delta_temp, factor_precio, etiqueta):
    sim = df_base.copy()

    # Aplicar supuestos del escenario
    sim['Rainfall(mm)'] = sim['Rainfall(mm)'] * factor_lluvia
    sim['Temperature(C°)'] = sim['Temperature(C°)'] + delta_temp
    sim['price'] = sim['price'] * factor_precio

    # Preparar columnas exactamente en el mismo orden que X_final
    X_sim = pd.get_dummies(sim.drop(columns=['price']), drop_first=True)
    X_sim = X_sim.reindex(columns=X_final.columns, fill_value=0)

    # Predecir rendimiento (revertir log1p) y calcular ingreso por hectárea
    sim[f'Yield_{etiqueta}'] = np.expm1(modelo_final_log.predict(X_sim))
    sim[f'Income_ha_{etiqueta}'] = sim[f'Yield_{etiqueta}'] * sim['price']

    return sim[['Location', 'Crops', 'Area(ha)', 'Irrigation', 'Soil type', f'Yield_{etiqueta}', f'Income_ha_{etiqueta}']]

# 1. Proyectar 2026 (Escenario Base)
proj_2026 = proyectar_escenario(linea_base, factor_lluvia=1.0, delta_temp=0.0, factor_precio=1.03, etiqueta='2026')

# 2. Proyectar 2027 (Escenario de Estrés Climático)
proj_2027 = proyectar_escenario(linea_base, factor_lluvia=0.85, delta_temp=1.5, factor_precio=1.06, etiqueta='2027')

# 3. Unir ambas proyecciones en una sola tabla comparativa
proyeccion_final = proj_2026.merge(
    proj_2027[['Location', 'Crops', 'Yield_2027', 'Income_ha_2027']],
    on=['Location', 'Crops']
)

print("=== TOP 10 ZONAS DE MAYOR RIESGO FINANCIERO PROYECTADO (2026 vs 2027) ===")
display(proyeccion_final.sort_values('Income_ha_2027').head(10))

=== TOP 10 ZONAS DE MAYOR RIESGO FINANCIERO PROYECTADO (2026 vs 2027) ===


      Location      Crops  Area(ha) Irrigation     Soil type  Yield_2026  \
46  Kasaragodu      Cocoa   17549.5       Drip     clay loam    0.026722   
44  Kasaragodu   Cardamum    3467.0       Drip          clay    0.432832   
77   Mangalore     Pepper   15009.0       Drip      laterite    0.115623   
60    Madikeri   Cardamum   13283.0       Drip  drained loam    0.181310   
34      Hassan      Cocoa   10922.0       Drip      alluvial    0.168016   
88     Raichur  Blackgram   10911.0       Drip         black    0.139458   
43  Kasaragodu  Blackgram    5876.0       Drip    heavy clay    0.192734   
45  Kasaragodu     Cashew    3649.0       Drip    heavy clay    0.068847   
38      Hassan  Groundnut   14249.0      Spray  well drained    0.114980   
13   Davangere  Blackgram   10922.0       Drip    heavy clay    0.211175   

    Income_ha_2026  Yield_2027  Income_ha_2027  
46     1409.842962    0.041877     2273.762340  
44     5708.463221    0.480216     6517.862485  
77     5381.2955

In [30]:
# Primeras visualizaciones del final de ejercicio

# 1. Definir tipo de cambio (INR a USD) y crear columnas convertidas
TASA_INR_USD = 88.0

df_limpio['price_USD'] = df_limpio['price'] / TASA_INR_USD
df_limpio['Income_Total_USD'] = df_limpio['Income_Total'] / TASA_INR_USD
df_limpio['Income_per_ha_USD'] = df_limpio['Income_per_ha'] / TASA_INR_USD

proyeccion_final['Income_USD_ha_2026'] = proyeccion_final['Income_ha_2026'] / TASA_INR_USD
proyeccion_final['Income_USD_ha_2027'] = proyeccion_final['Income_ha_2027'] / TASA_INR_USD

# 2. Gráficas históricas en USD (Cultivos y Zonas de Riesgo)
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

peores_cultivos_usd = (
    df_limpio.groupby('Crops')['Income_per_ha_USD']
    .median()
    .sort_values(ascending=True)
    .reset_index()
)
sns.barplot(data=peores_cultivos_usd, x='Income_per_ha_USD', y='Crops', ax=axes[0], palette='Reds_r')
axes[0].set_title('Ingreso Mediano por Hectárea según Cultivo (USD)')
axes[0].set_xlabel('Ingreso por Hectárea (USD / ha)')
axes[0].set_ylabel('Cultivo')

zonas_riesgo_usd = (
    df_limpio.groupby(['Location', 'Crops'])['Income_per_ha_USD']
    .median()
    .sort_values(ascending=True)
    .head(10)
    .reset_index()
)
zonas_riesgo_usd['Zona_Cultivo'] = zonas_riesgo_usd['Location'] + ' - ' + zonas_riesgo_usd['Crops']
sns.barplot(data=zonas_riesgo_usd, x='Income_per_ha_USD', y='Zona_Cultivo', ax=axes[1], palette='Oranges_r')
axes[1].set_title('Top 10 Zonas de Alto Riesgo Histórico (USD / ha)')
axes[1].set_xlabel('Ingreso por Hectárea (USD / ha)')
axes[1].set_ylabel('Ubicación - Cultivo')

plt.tight_layout()
plt.show()

# 3. Gráfica de Proyección 2026 vs 2027 en USD
top10_riesgo = proyeccion_final.sort_values('Income_USD_ha_2027').head(10).copy()
top10_riesgo['Zona_Cultivo'] = top10_riesgo['Location'] + ' - ' + top10_riesgo['Crops']

y_pos = np.arange(len(top10_riesgo))
alto = 0.38

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(y_pos - alto/2, top10_riesgo['Income_USD_ha_2026'], height=alto, label='2026 (Escenario Base)', color='#4C72B0')
ax.barh(y_pos + alto/2, top10_riesgo['Income_USD_ha_2027'], height=alto, label='2027 (Estrés Climático)', color='#DD8452')

ax.set_yticks(y_pos)
ax.set_yticklabels(top10_riesgo['Zona_Cultivo'])
ax.set_xlabel('Ingreso Proyectado por Hectárea (USD / ha)')
ax.set_title('Top 10 Zonas de Mayor Riesgo Financiero: Proyección 2026 vs 2027 (USD)')
ax.legend()
ax.grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

/tmp/ipykernel_11016/41769797.py:23: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=peores_cultivos_usd, x='Income_per_ha_USD', y='Crops', ax=axes[0], palette='Reds_r')
/tmp/ipykernel_11016/41769797.py:36: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=zonas_riesgo_usd, x='Income_per_ha_USD', y='Zona_Cultivo', ax=axes[1], palette='Oranges_r')


<Figure size 1600x600 with 2 Axes>

<Figure size 1200x600 with 1 Axes>

In [40]:
# Arreglando hoja anterior:

import plotly.express as px

# Reconstruir df_mapa con las 11 ubicaciones de Karnataka a partir de proyeccion_final
coordenadas_karnataka = {
    'Bangalore': (12.9716, 77.5946),
    'Chikmangaluru': (13.3161, 75.7720),
    'Davangere': (14.4644, 75.9218),
    'Gulbarga': (17.3297, 76.8343),
    'Hassan': (13.0033, 76.1004),
    'Kasaragodu': (12.4996, 74.9869),
    'Kodagu': (12.3375, 75.8069),
    'Madikeri': (12.4244, 75.7382),
    'Mangalore': (12.9141, 74.8560),
    'Mysuru': (12.2958, 76.6394),
    'Raichur': (16.2076, 77.3463),
}

df_mapa = (
    proyeccion_final.groupby('Location')
    .agg(
        Med_USD_2026=('Income_USD_ha_2026', 'median'),
        Med_USD_2027=('Income_USD_ha_2027', 'median'),
        Peor_Cultivo=('Crops', lambda s: proyeccion_final.loc[s.index].sort_values('Income_USD_ha_2027').iloc[0]['Crops']),
        Mejor_Cultivo=('Crops', lambda s: proyeccion_final.loc[s.index].sort_values('Income_USD_ha_2027').iloc[-1]['Crops']),
    )
    .reset_index()
)
df_mapa['Lat'] = df_mapa['Location'].map(lambda x: coordenadas_karnataka[x][0])
df_mapa['Lon'] = df_mapa['Location'].map(lambda x: coordenadas_karnataka[x][1])
df_mapa['Dolor_2027'] = 1 / df_mapa['Med_USD_2027'].clip(lower=1)

# Mapamundi interactivo con las áreas de dolor proyectadas para 2027
fig = px.scatter_geo(
    df_mapa,
    lat='Lat',
    lon='Lon',
    color='Med_USD_2027',
    size='Dolor_2027',
    hover_name='Location',
    hover_data={
        'Lat': False,
        'Lon': False,
        'Med_USD_2026': ':$.2f',
        'Med_USD_2027': ':$.2f',
        'Peor_Cultivo': True,
        'Mejor_Cultivo': True,
        'Dolor_2027': False
    },
    color_continuous_scale='RdYlGn',  # Rojo = mayor dolor financiero, Verde = mayor ingreso
    size_max=28,
    projection='natural earth',
    title='Mapamundi de Riesgo Agrícola 2027: Áreas de Dolor Financiero (USD/ha)'
)

fig.update_geos(
    showcountries=True, countrycolor='#888888',
    showcoastlines=True, coastlinecolor='#555555',
    showland=True, landcolor='#F4F4F2',
    showocean=True, oceancolor='#E6F0F8',
    fitbounds=False  # Mantiene la vista completa del mapamundi; puedes hacer zoom con el mouse
)

fig.update_layout(
    height=600,
    coloraxis_colorbar_title='Ingreso 2027<br>(USD/ha)'
)

fig.show()

# ESTRATEGIA PARA SOLVENTAR PROBLEMÁTICA

# 1. Obtener el área modelo (mediana del Top 20% mejor pagado) por Cultivo y Tipo de Suelo
area_optima_df = (
    df_limpio.sort_values('Income_per_ha_USD', ascending=False)
    .groupby(['Crops', 'Soil type'])
    .apply(lambda g: g.head(max(1, int(np.ceil(len(g) * 0.20))))['Area(ha)'].median(), include_groups=False)
    .reset_index(name='Area_Optima_ha')
)

# 2. Preparar simulación 2027 (Estrés climático: -15% lluvia, +1.5°C, +6% precio)
sim_2027 = linea_base.copy()
sim_2027['Rainfall(mm)'] = sim_2027['Rainfall(mm)'] * 0.85
sim_2027['Temperature(C°)'] = sim_2027['Temperature(C°)'] + 1.5

# CORRECCIÓN: Definir 'price_USD' a partir de 'price' antes de aplicar el factor de incremento
TASA_INR_USD = 88.0
sim_2027['price_USD'] = (sim_2027['price'] / TASA_INR_USD) * 1.06

# Predicción 2027 sin seccionar (Área actual)
X_2027_actual = pd.get_dummies(sim_2027.drop(columns=['price', 'price_USD']), drop_first=True)
X_2027_actual = X_2027_actual.reindex(columns=X_final.columns, fill_value=0)
sim_2027['Income_2027_Actual_USD'] = np.expm1(modelo_final_log.predict(X_2027_actual)) * sim_2027['price_USD']

# 3. Aplicar seccionamiento según Cultivo + Tipo de Suelo
sim_2027 = sim_2027.merge(area_optima_df, on=['Crops', 'Soil type'], how='left')
sim_2027['Sub_Parcelas_UMI'] = np.ceil(sim_2027['Area(ha)'] / sim_2027['Area_Optima_ha']).fillna(1).astype(int)

sim_seccionado = sim_2027.copy()
sim_seccionado['Area(ha)'] = sim_seccionado['Area_Optima_ha'].fillna(sim_seccionado['Area(ha)'])

X_2027_sec = pd.get_dummies(sim_seccionado[linea_base.columns].drop(columns=['price']), drop_first=True)
X_2027_sec = X_2027_sec.reindex(columns=X_final.columns, fill_value=0)
sim_2027['Income_2027_Seccionado_USD'] = np.expm1(modelo_final_log.predict(X_2027_sec)) * sim_2027['price_USD']

# Escenario realista: ganancia por eficiencia operativa con rendimientos decrecientes
ELASTICIDAD_UMI = 0.35
ratio_area = (sim_2027['Area(ha)'] / sim_2027['Area_Optima_ha']).clip(lower=1.0)
factor_mejora = 1 + ELASTICIDAD_UMI * np.log(ratio_area)

# Limitamos el ingreso realista para que nunca supere el techo teórico del modelo
sim_2027['Income_2027_Realista_USD'] = np.minimum(
    sim_2027['Income_2027_Actual_USD'] * factor_mejora,
    sim_2027['Income_2027_Seccionado_USD']
)

# 4. Mostrar comparativa limpia (Actual vs Meta Realista por Seccionamiento UMI)
estrategia_final = sim_2027.sort_values('Income_2027_Actual_USD').head(10)[
    ['Location', 'Crops', 'Area(ha)', 'Area_Optima_ha', 'Sub_Parcelas_UMI',
     'Income_2027_Actual_USD', 'Income_2027_Realista_USD']
]

print("\n----------------------Estrategia Final-----------------------\n")
print(estrategia_final.round(2).to_string(index=False))


----------------------Estrategia Final-----------------------

  Location     Crops  Area(ha)  Area_Optima_ha  Sub_Parcelas_UMI  Income_2027_Actual_USD  Income_2027_Realista_USD
Kasaragodu     Cocoa   17549.5           153.0               115                   25.84                     68.72
Kasaragodu  Cardamum    3467.0           103.0                34                   74.07                    165.22
 Mangalore    Pepper   15009.0          2578.0                 6                   75.81                    122.55
  Madikeri  Cardamum   13283.0           596.5                23                   90.47                    188.73
    Hassan     Cocoa   10922.0           161.5                68                   91.81                    227.22
   Raichur Blackgram   10911.0           148.0                74                   92.92                    232.78
Kasaragodu Blackgram    5876.0           206.0                29                  102.42                    222.54
Kasaragodu    Ca